In [ ]:
# ruff: noqa: F405, F403, E402, F811
# no cache
%load_ext autoreload
%autoreload 2

In [ ]:
# doc-hidden
import os
import sys
import mediapy
import numpy as np

# put the local folder at *the very front*
# The kernel CWD may differ from notebook location, so find src relative to project root
module_path = os.path.abspath("src") if os.path.exists("src/ptspy") else os.path.abspath("../src")

if module_path not in sys.path:
    sys.path.insert(0, module_path)  # use insert(0,…), not append

# throw away the already-imported wheel copy
for m in list(sys.modules):
    if m == "ptspy" or m.startswith("ptspy."):
        del sys.modules[m]

import ptspy  # now comes from ../src

# print(ptspy.__file__)  # sanity-check

In [ ]:
from ptspy import *
import ipywidgets as widgets

### Make a point

Begin from the beginning, let's begin with a point.

In [ ]:
@frame()
def draw(form):
    form.fillOnly("000").point(form.size / 2, 10)


draw()

Ptspy makes things simple. To create an image, you start with a function that has `form` as the first parameter, which is an instance of the `Form` class to simplify code for drawing. Then, decorate the function with `@frame` and call the function, and you have an image. 

Let's customize the frame and add some parameters to the drawing function.

In [ ]:
@frame(bg="000", size=(300, 300))
def draw(form, radius):
    form.fillOnly("f0f3f7").point(form.size / 2, radius)


draw(100)

Here we set the frame's dimensions and color by passing parameters to `@frame`, and add a radius parameter to our custom function. That's it!

This means you can also use UI widgets to make it interactive for testing, with just one more function.

In [ ]:
@frame()
def draw(form, radius):
    form.fillOnly("000").point(form.size / 2, radius)


# interactive UI via ipywidgets
# you may need to run `jupyter labextension enable widgetsnbextension` first
@widgets.interact(radius=(10, 100))
def ui(radius):
    return draw(radius)

Making an animation is just as easy. Instead of `@frame`, we'll use `@frames` and render a number of frames. In our draw function, we'll add a new parameter `index` to get the current frame index. 

In [ ]:
@frames(numFrames=60)
def draw(form, index, radius):
    form.fillOnly("000").point(form.size / 2, radius + index)


video = draw(radius=10)
mediapy.show_video(video, codec="h264", fps=30)

### Connect the dots

A point is the simplest representation of a concept. The meaning of a point depends on the definition of a space.

To create points in n-dimensional space, use `P` as a shorthand for `np.array` function.

In [ ]:
P(1, 2, 3)

`P` is short and flexible. It can take lists, tuples, or floats and returns an array of float values to represent points in space.

In [ ]:
P([(1, 2), (3, 4)])

`P2` is a shortcut for creating a grid of 2D points, and `P3` will create a cube of 3D points.

In [ ]:
grid = P2([1, 2, 3], [4, 5, 6])
print(pretty(grid))

Numpy is used as default, so you have access to all the numpy features. You can also swap it to another numpy-like library, like cupy.

In [ ]:
points = grid.reshape(-1, 2).T
print(pretty(points))

Pts provides some simple functions to remove frictions and clutters and helps you think clearly and connect the dots.

`pretty`, as you've seen above, convert the array to a clearly-formatted string.

In [ ]:
print(pretty(P([[1, 2], [3, 4]])), "\n\nvs default:\n", np.array([[1.0, 2.0], [3.0, 4.0]]))

Quickly make a series for testing with `populate`. Take a look at the `compose` package for other ways to create structures.

In [ ]:
populate(10, 0.1, 5)

`random` is a handy function to generate a single random number, or a range of uniformly distributed numbers.

In [ ]:
print(pretty(random((3, 2))))

`randomizer` returns a random generator. Use it to access the random methods and distribution options in numpy. 

Let's visualize a simple shuffling of numbers.

In [ ]:
@frames(bg="000", size=(300, 150), numFrames=30)
def draw(form, index, rand):
    row = P([50, 50], [100, 50], [150, 50], [200, 50], [250, 50])
    row += P(-8, 5)  # adjust position
    values = populate(row.shape[0], 1, 1)

    row2 = np.copy(row) + P(0, 50)
    shuffled = np.copy(values)
    rand.shuffle(shuffled)

    form.font(24)

    for pos, val, pos2, val2 in zip(row, values, row2, shuffled):
        form.fillOnly("fff").text(pos, str(int(val)))
        form.fillOnly("fff").text(pos2, str(int(val2)))


video = draw(rand=randomizer())
mediapy.show_video(video, codec="h264", fps=10)

### Find the right colors

Color spaces are unintuitive and color choices are subjective. How do we make it easier?

Letting go of technical jargons and preconceptions, an intuitive way to think about color can be: what the hue is, how vivid it is, and how light it is. This maps intuitively to a 3D cube with 3 axes: hue, chroma, lightness.

Here we aim for intuition over perfection - Decide how many steps you need in each dimension, make a cube with `colorCube()`, and slice it to get a palette. 

In [ ]:
@frame(size=(300, 300))
def draw(form, steps):
    cube = colorCube(steps)
    slice = cube[:, 14, :]  # take the middle chroma slice (color intensity)
    form.image(slice, P([0, 0], form.size))


draw(steps=25)

`colorPicker()` can retrieve a color in a color cube. Using values between 0 to 1, you can easily get to nuanced color variations.

In [ ]:
@frame(size=(300, 300))
def draw(form, steps):
    cube = colorCube(steps)
    form.fillOnly(colorPicker(cube, 0.20, 0.9, 0.7)).rect(P([[0, 0], form.size]))
    form.fillOnly(colorPicker(cube, 0.18, 0.9, 0.8)).rect(P([form.size * 0.25, form.size * 0.75]))


draw(30)

For an even simpler starting point, apply `colorScheme` to your `colorCube`. This gives you a practical color palette with consistent steps from dark to light.

The `intensity` parameter (0 to 1) adjusts both lightness and vividity as a whole. If you passed a random generator to `variant`, the colors will vary by random each time.

In [ ]:
@frame(bg="#00000000", size=(450, 300))
def draw(form, cubeSize=24, intensity=0.5, hues=9, swatches=6, variantSeed=0):
    cube = colorCube(steps=cubeSize)
    variant = randomizer(variantSeed)
    palette = colorScheme(cube, hues, swatches, intensity, variant)
    form.colorMap(palette[:, :, 0], P([0, 0], form.size), style="circle")


draw(variantSeed=8)  # random variant wtih seed 8

Another use of the `colorCube` is to interpolate between two colors. We apply `colorInterpolation` to get the indices between two color swatches.

You can interpolate them as linear paths:

In [ ]:
@frame(bg="#00000000", size=(300, 300))
def draw(form):
    cc = colorCube(steps=100)
    # batch 2 interpolations in one call
    startColors = [P(80, 28, 99), P(50, 90, 50)]  # hue, chroma, lightness
    endColors = [P(10, 99, 70), P(80, 30, 10)]
    indices = colorInterpolation(cc, start=startColors, end=endColors, steps=8, asPath=True)
    form.colorMap(indices, P([0, 0], form.size))


draw()

Or a 2D slice from the color cube:

In [ ]:
@frame(size=(300, 300))
def draw(form, lightness):
    cc = colorCube(steps=100)

    indices = colorInterpolation(cc, P(20, 28, 99), P(50, 99, 10), steps=20, asPath=False)

    # You can also use colorPicker helper to get colors from normalized values
    # picker = colorPicker(cc, asArray=True)
    # indices = colorInterpolation(
    #     cc, picker(0.95, 0.28, 0.2), picker(0.1, 0.99, 0.5), steps=12, asPath=False
    # )

    first = np.squeeze(indices, axis=0)
    form.image(first[:, :, lightness].T, P([0, 0], form.size))


draw(lightness=5)

`gradient()` is a convenient function to create gradients directly. It supports color stops and 4 different styles: linear, radial, sweep, and conical.

In [ ]:
@frame(size=(300, 300))
def draw(form):
    colors = [0xFF2099FF, 0xFFFF66FF, 0x20CCE0FF]  # ARGB
    shader = gradient(P([[100, 100]]), colors, "sweep", stops=[0.2, 0.5, 0.8])
    form.fill(shader).rect(P([[0, 0], form.size]))


draw()

### Form follows function()

A vision appears. It twinkles like tiny stars in our imagination. As we connect the dots, it scaffolds into a mallable form. We squeeze it, stretch it, poke it, pat it. It hardens into something tangible. We put a beautiful glaze on it and set it free.

Can code be a new substrate of imagination? How might we compose, process, and visualize ideas on it? Will form follow function()? 

How do we make a visual structure in code? We use functions to compose it, and we use functions to change it. Instead of thinking inputs and outputs, we can think of it as the relationship of possibilities, as a shortcut between 2 forms in our imagination. 

Here's a simple example of distributing points on a path, using the `steps` function.

In [ ]:
@frame(bg="000", size=(300, 300))
def draw(form):
    points = steps([40, 150], [form.width - 40, 150], count=20, shaping="cubicOut")
    form.fillOnly("f0f5fa").points(points, 2)


draw()

What if we want to distribute points, but don't know where it'll go and how many there will be? An incomplete function returns a new function, rescoping the range of possibilities.

Here the `step` function starts out incomplete and commits later on in the animation loop.

In [ ]:
# you can skip positional argument
# or use TBD token to indicate parameter placeholder (eg, count=TBD)
pointsFn = steps([40, 150], shaping="cubicOut")


@frames(bg="000", size=(300, 300), numFrames=60)
def draw(form, index):
    count = int(index / 2.0)
    end = [form.width - 40, 40 + index / 60.0 * (form.height - 80)]
    form.fillOnly("f0f5fa").points(pointsFn(end, count), 2)


video = draw()
mediapy.show_video(video, codec="h264", fps=30)

Most of the functions in ptspy can be applied partially like the above. There's also a convenient `@partial` decorator to define your own partials. 

Here we create a 2 partial functions `first()` and `second()` from `colorDisc()`, and use them to draw two color discs.

In [ ]:
@partial(placeholder=TBD)
def createPolygon(form, center, sides, rotation):
    poly = regularPolygon(center=center * form.size, radius=form.width / 6, sides=sides, closed=True)
    form.save()
    form.rotate(rotation, pivot=center * form.size)
    form.fillOnly("fff", 2).polygon(poly)
    form.restore()


# create 4 polygons in a grid but set size and rotation later
centers = P([0.33, 0.33], [0.66, 0.33], [0.33, 0.66], [0.66, 0.66])
polygons = [createPolygon(form=TBD, center=c, sides=s) for c, s in zip(centers, [4, 5, 6, 7])]


@frames(bg="000", size=(300, 300), numFrames=90)
def draw(form, index):
    for poly in polygons:
        poly(form=form, rotation=index * 4)


video = draw()
mediapy.show_video(video, codec="h264", fps=30)

Partial function may feel like a simple intuition, but it holds a deeper insight about creative work. Outcomes converge and diverge with each parameter you set, each function you apply. We wander in an infinite field of possibilities, seeking hidden relationships between disparate ideas. 

Another common functional practice is that of a pipeline, often confusingly called `pipe` and `compose`, but the idea is simple enough: Compose a chain of functions so that the output of one function becomes the input into another function. This creates reusable transformation sequences that are easy to read and modify. 

In [ ]:
def a(x, n):
    return P(x, x * n)


def b(p):
    # return a tuple to pass multiple args to next function
    return (P(p, p + 100), 5)


def c(path, repeat):
    return steps(path[0], path[1], repeat)


# You can also write this using syntax: `F(a) >> b >> c`
# or reversePipeline as: `F(a) << b << c`
fn = pipeline(a, b, c)
fn(10, 2)

There are two ways to build pipelines:
  - pipeline(a, b, c) — chain functions with a function call
  - F(a) >> b >> c — chain functions with operators (same result, different syntax)

This means we can deconstruct a complex shape into simpler and reusable components, and reconstruct them in various ways. We are the puzzle maker and the puzzle player.

In [ ]:
makePolygon = regularPolygon(TBD, sides=4, radius=40, closed=True)


def getNext(form, pts):
    results = []
    for pt in pts:
        next = makePolygon(pt)
        form.strokeOnly("f0f5fa", 1).lines(next)
        results.append(next)
    # auto-unpack tuples as parameters for next function
    return (form, np.vstack(results))


def makeDot(form, pts):
    form.fillOnly("fff").points(pts, 3)
    return (form, pts)


# same as pipeline(getNext, makeDot, ...)
makeAll = F(getNext) >> makeDot >> getNext


@frame(bg="000", size=(300, 300))
def draw(form):
    poly = makePolygon([150, 150])
    form.strokeOnly("f0f5fa", 1).lines(poly)
    makeAll(form, poly)


draw()

Now for a more complex example, you can use <code>lazy=True</code> to get an array of generator functions of each intermediate step. This can be useful for debugging and animation.

In [ ]:
makePolygon = regularPolygon(TBD, TBD, radius=30, closed=True)


def getNext(pts, sides):
    if len(pts.shape) > 2:
        pts = pts.reshape(-1, 2)
    return (P([makePolygon(center, sides) for center in pts]), sides)


def makeDot(pts, sides):
    return (pts.reshape(-1, 2), sides)


# get intermediate steps in the pipeline as a generator, using lazy=True
makeAll = pipeline(getNext, getNext, getNext, makeDot, lazy=True)
progress = list(makeAll(P(150, 150), 6))  # type: ignore (or cast to Generator if preferred)


@frames(bg="000", size=(300, 300), numFrames=20)
def draw(form, index):
    i = index % len(progress)
    if len(progress[i][0].shape) > 2:
        form.strokeOnly("f0f5fa").lines(progress[i][0].reshape(-1, 2))
    else:
        form.fillOnly("f0f5fa").points(progress[i][0], 2)


video = draw()
mediapy.show_video(video, codec="h264", fps=2)

Pipeline supports additional features such as async and trace. Also look for other convenient functions in `process` and `compose` modules. Let your imagination connect the dots!

In [ ]:
@frames(bg="000", size=(300, 300), numFrames=60)
def draw(form, index):
    top = steps(0, form.width, 20, shaping=lambda t, a, b: a + (b - a) * (t * t * index / 30.0))
    top = expand(top[:, np.newaxis], 0)  # reshape with new axis and append 0. shape is now (n,2)
    lines = np.stack([top, top + [0, form.height]], axis=1)

    form.strokeOnly("f0f5fa", 10 - index / 10.0).lines(lines)


video = draw()
mediapy.show_video(video, codec="h264", fps=30)